# Convert GlycoMotif JSON to an RDFLib graph

This notebook loads the supplied `motif_detail_GGM.000001.json` record, converts its nested JSON structure into RDF using **RDFLib**, validates the result, and exports it as Turtle and JSON-LD.

The conversion is intentionally schema-flexible: JSON object fields become RDF predicates, nested objects and object-array entries become blank nodes, and scalar values become typed RDF literals. URL-valued fields are represented as `URIRef` objects. The original JSON is preserved structurally; no glycan chemistry is inferred from the sequence strings.

## 2. Imports and input path

Place the JSON file beside this notebook, or change `JSON_PATH` to its location.

In [1]:
from pathlib import Path
import json
import re
from urllib.parse import urlparse

from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, XSD

JSON_PATH = Path("motif_detail_GGM.000001.json")
OUTPUT_TTL = Path("motif_detail_GGM.000001.ttl")
OUTPUT_JSONLD = Path("motif_detail_GGM.000001.jsonld")

with JSON_PATH.open("r", encoding="utf-8") as f:
    record = json.load(f)

print(f"Loaded {len(record)} top-level fields")
print("Motif accession:", record.get("motif_ac", "(not provided)"))

Loaded 30 top-level fields
Motif accession: GGM.000001


## 3. Define the RDF vocabulary and conversion rules

- The motif is the root resource, identified by its `motif_ac` when available.
- Predicates use a local namespace and the JSON key as their local name.
- Nested dictionaries and dictionaries inside lists are represented as blank nodes.
- Lists preserve every entry, including repeated scalar values.
- Boolean, integer, float, and string values become appropriately typed literals.
- Fields named `url` or ending in `_url` become URI references when they contain valid HTTP(S) URLs; otherwise they remain literals.

In [2]:
BASE = Namespace("https://glycomotif.glyomics.org/rdf/")
MOTIF_BASE = "https://glycomotif.glyomics.org/glycomotif/"

g = Graph()
g.bind("gm", BASE)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)

def safe_local_name(value):
    """Make a stable RDF local name from a JSON key."""
    name = re.sub(r"[^A-Za-z0-9_]", "_", str(value))
    if not name or not re.match(r"[A-Za-z_]", name):
        name = "field_" + name
    return name

def is_http_url(value):
    if not isinstance(value, str):
        return False
    parsed = urlparse(value)
    return parsed.scheme in {"http", "https"} and bool(parsed.netloc)

def scalar_to_node(value, field_name=None):
    """Convert a JSON scalar to an RDF term."""
    if value is None:
        return None
    if isinstance(value, bool):
        return Literal(value, datatype=XSD.boolean)
    if isinstance(value, int):
        return Literal(value, datatype=XSD.integer)
    if isinstance(value, float):
        return Literal(value, datatype=XSD.double)
    if isinstance(value, str):
        if field_name and (field_name == "url" or field_name.endswith("_url")) and is_http_url(value):
            return URIRef(value)
        return Literal(value, datatype=XSD.string)
    return Literal(str(value))

def add_json_value(graph, subject, predicate, value, field_name=None):
    """Recursively add one JSON value to the graph."""
    if value is None:
        return

    if isinstance(value, dict):
        node = BNode()
        graph.add((subject, predicate, node))
        for key, child_value in value.items():
            child_predicate = BASE[safe_local_name(key)]
            add_json_value(graph, node, child_predicate, child_value, field_name=key)
        return

    if isinstance(value, list):
        # Each list element is attached through the same predicate.
        for item in value:
            add_json_value(graph, subject, predicate, item, field_name=field_name)
        return

    obj = scalar_to_node(value, field_name=field_name)
    if obj is not None:
        graph.add((subject, predicate, obj))

motif_id = record.get("motif_ac")
if motif_id:
    motif_uri = URIRef(MOTIF_BASE + str(motif_id))
else:
    motif_uri = URIRef("urn:glycomotif:record:motif-detail")

g.add((motif_uri, RDF.type, BASE.GlycanMotif))
for key, value in record.items():
    if key == "motif_ac":
        # Keep the accession explicitly as a literal as well as in the URI.
        g.add((motif_uri, BASE.motif_ac, Literal(str(value))))
    else:
        add_json_value(g, motif_uri, BASE[safe_local_name(key)], value, field_name=key)

print("RDF triples:", len(g))
print("Root resource:", motif_uri)

RDF triples: 84783
Root resource: https://glycomotif.glyomics.org/glycomotif/GGM.000001


## 4. Inspect the graph

The first few triples are shown for a quick sanity check. The source record contains a large `glycans` list, so the graph may contain many triples.

In [3]:
for triple in list(g)[:20]:
    print(triple)

(rdflib.term.BNode('Nad04fe1448234b5798fcecff24a0a252'), rdflib.term.URIRef('https://glycomotif.glyomics.org/rdf/glytoucan_ac'), rdflib.term.Literal('G48064BJ', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('https://glycomotif.glyomics.org/glycomotif/GGM.000001'), rdflib.term.URIRef('https://glycomotif.glyomics.org/rdf/glycans'), rdflib.term.BNode('N13c803999db54dc09f9618690e5cdd05'))
(rdflib.term.BNode('N42aff73c9437461ba557552e57273463'), rdflib.term.URIRef('https://glycomotif.glyomics.org/rdf/glytoucan_ac'), rdflib.term.Literal('G71063DT', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.BNode('N329171c78cd24d5e967b69adb174ce8c'), rdflib.term.URIRef('https://glycomotif.glyomics.org/rdf/url'), rdflib.term.URIRef('https://glytoucan.org/Structures/Glycans/G33653GT'))
(rdflib.term.BNode('N21ee1378f7484e598b0b8944a7ee3da3'), rdflib.term.URIRef('https://glycomotif.glyomics.org/rdf/url'), rdflib.term.URIRef

## 5. Run a small SPARQL query

This example retrieves the motif's main identifiers and structural representations. It uses the original JSON field names as RDF predicates.

In [4]:
query = f''' 
PREFIX gm: <{BASE}>
SELECT ?field ?value
WHERE {{
  VALUES ?field {{ gm:motif_ac gm:iupac gm:wurcs gm:glycoct gm:glycam }}
  <{motif_uri}> ?field ?value .
}}
ORDER BY ?field
'''
for row in g.query(query):
    print(row.field, "=>", row.value)

https://glycomotif.glyomics.org/rdf/glycam => DGalpb1-4DGlcpNAcb1-OH
https://glycomotif.glyomics.org/rdf/glycoct => RES 1b:b-dglc-HEX-1:5 2s:n-acetyl 3b:b-dgal-HEX-1:5 LIN 1:1d(2+1)2n 2:1o(4+1)3d
https://glycomotif.glyomics.org/rdf/iupac => beta-D-Galp-(1->4)-beta-D-GlcpNAc-(1->
https://glycomotif.glyomics.org/rdf/motif_ac => GGM.000001
https://glycomotif.glyomics.org/rdf/wurcs => WURCS=2.0/2,2,1/[a2122h-1b_1-5_2*NCC/3=O][a2112h-1b_1-5]/1-2/a4-b1


## 6. Export the graph

Turtle is a compact, human-readable RDF serialization. JSON-LD is useful when integrating with JSON-oriented linked-data tools.

In [5]:
g.serialize(destination=str(OUTPUT_TTL), format="turtle")
g.serialize(destination=str(OUTPUT_JSONLD), format="json-ld", indent=2)

print(f"Saved {OUTPUT_TTL.resolve()}")
print(f"Saved {OUTPUT_JSONLD.resolve()}")
print(f"Graph contains {len(g):,} triples")

Saved /home/marta/Documents/valvian/motif_detail_GGM.000001.ttl
Saved /home/marta/Documents/valvian/motif_detail_GGM.000001.jsonld
Graph contains 84,783 triples


## 7. Reload and validate the Turtle export

Parse the exported file into a fresh graph and confirm that the triple count is unchanged.

In [6]:
check_graph = Graph()
check_graph.parse(str(OUTPUT_TTL), format="turtle")

assert len(check_graph) == len(g), (
    f"Triple count changed: original={len(g)}, reloaded={len(check_graph)}"
)
print("Validation passed:", len(check_graph), "triples successfully reloaded.")

Validation passed: 84783 triples successfully reloaded.
